# Sphere Dataset Builder (Netgen-Enforced + Tet-Count Controlled)

"
"This notebook generates dataset pairs (`T_bad`, `T_good_cell`) with the following guarantees:
"
"- Good mesh is optimized with **Netgen** (required per accepted sample).
"
"- Tet count is constrained to target tolerance (default ±5%).
"
"- Node positions are diversified via randomized embedded-point distributions/deformations,
"
"  without allowing large tet-count variance.


In [47]:
import gc
import os
import sys
import time
import tempfile
import subprocess
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


In [48]:
def init_mat73_stream_file(out_path, n_samples):
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    f = h5py.File(out_path, 'w')
    f.create_dataset('P', shape=(n_samples, 1), dtype=h5py.ref_dtype)
    f.create_dataset('T_bad', shape=(n_samples, 1), dtype=h5py.ref_dtype)
    f.create_dataset('T_good_cell', shape=(n_samples, 1), dtype=h5py.ref_dtype)
    f.create_dataset('minQ_bad', shape=(n_samples, 1), dtype=np.float64)
    f.create_dataset('minQ_good', shape=(n_samples, 1), dtype=np.float64)
    return f


def write_one_sample(f, i, P, T_bad_1based, T_good_1based, minQ_bad, minQ_good):
    dP = f.create_dataset(f'P_{i}', data=np.asarray(P, dtype=np.float64).T)
    dTb = f.create_dataset(f'T_bad_{i}', data=np.asarray(T_bad_1based, dtype=np.int32).T)
    dTg = f.create_dataset(f'T_good_{i}', data=np.asarray(T_good_1based, dtype=np.int32).T)

    f['P'][i, 0] = dP.ref
    f['T_bad'][i, 0] = dTb.ref
    f['T_good_cell'][i, 0] = dTg.ref
    f['minQ_bad'][i, 0] = float(minQ_bad)
    f['minQ_good'][i, 0] = float(minQ_good)


def write_params_group(f, params):
    if 'params' in f:
        del f['params']
    g = f.create_group('params')
    for k, v in params.items():
        if isinstance(v, str):
            dt = h5py.string_dtype(encoding='utf-8')
            g.create_dataset(k, data=np.array(v, dtype=dt))
        else:
            g.create_dataset(k, data=np.asarray(v))


In [49]:
def run_worker_sample(
    worker_path,
    *,
    seed,
    radius,
    mesh_size_start,
    target_nodes,
    node_tolerance,
    target_tets,
    tet_tolerance_frac,
    max_attempts,
    n_embedded_points,
    embedded_radius_frac,
    local_size_jitter_min,
    local_size_jitter_max,
    boundary_bias,
    deformation_strength,
    warp_affine_strength,
    warp_radial_strength,
    tet_quality_mode,
    optimize_methods_csv,
    timeout_sec,
):
    with tempfile.NamedTemporaryFile(suffix='.npz', delete=False) as tmp:
        tmp_path = tmp.name

    cmd = [
        sys.executable,
        str(worker_path),
        '--out', tmp_path,
        '--seed', str(int(seed)),
        '--radius', str(float(radius)),
        '--mesh-size-start', str(float(mesh_size_start)),
        '--target-nodes', str(int(target_nodes)),
        '--node-tolerance', str(int(node_tolerance)),
        '--target-tets', str(int(target_tets)),
        '--tet-tolerance-frac', str(float(tet_tolerance_frac)),
        '--max-attempts', str(int(max_attempts)),
        '--n-embedded-points', str(int(n_embedded_points)),
        '--embedded-radius-frac', str(float(embedded_radius_frac)),
        '--local-size-jitter-min', str(float(local_size_jitter_min)),
        '--local-size-jitter-max', str(float(local_size_jitter_max)),
        '--boundary-bias', str(float(boundary_bias)),
        '--deformation-strength', str(float(deformation_strength)),
        '--warp-affine-strength', str(float(warp_affine_strength)),
        '--warp-radial-strength', str(float(warp_radial_strength)),
        '--tet-quality-mode', str(tet_quality_mode),
        '--optimize-methods', str(optimize_methods_csv),
    ]

    try:
        cp = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=float(timeout_sec),
            check=False,
        )

        if cp.returncode != 0:
            sig = -cp.returncode if cp.returncode < 0 else None
            raise RuntimeError(
                'worker_failed rc={} sig={} stdout_tail={} stderr_tail={}'.format(
                    cp.returncode,
                    sig,
                    cp.stdout[-800:],
                    cp.stderr[-1400:],
                )
            )

        with np.load(tmp_path) as z:
            return {
                'P': z['P'],
                'T_bad': z['T_bad'],
                'T_good': z['T_good'],
                'minQ_bad': float(z['minQ_bad'][0]),
                'minQ_good': float(z['minQ_good'][0]),
                'num_nodes': int(z['num_nodes'][0]),
                'num_tets_good': int(z['num_tets_good'][0]),
                'num_tets_bad': int(z['num_tets_bad'][0]),
                'netgen_used': bool(int(z['netgen_used'][0])),
                'matched': bool(int(z['matched'][0])),
                'attempts': int(z['attempts'][0]),
                'mesh_size_final': float(z['mesh_size_final'][0]),
            }
    finally:
        try:
            os.remove(tmp_path)
        except FileNotFoundError:
            pass


In [50]:
def calibrate_count_targets(
    *,
    worker_path,
    radius,
    mesh_size_start,
    tet_quality_mode,
    optimize_methods_csv,
    profile,
    timeout_sec,
    seed,
    n_calib=16,
):
    rng = np.random.default_rng(int(seed) + 12345)
    node_counts = []
    tbad_counts = []
    netgen_flags = []

    for _ in range(int(n_calib)):
        sample_seed = int(rng.integers(1, 2_000_000_000))
        s = run_worker_sample(
            worker_path,
            seed=sample_seed,
            radius=radius,
            mesh_size_start=mesh_size_start,
            target_nodes=0,
            node_tolerance=10**9,
            target_tets=0,
            tet_tolerance_frac=1.0,
            max_attempts=2,
            n_embedded_points=profile['n_embedded_points'],
            embedded_radius_frac=profile['embedded_radius_frac'],
            local_size_jitter_min=profile['local_size_jitter_min'],
            local_size_jitter_max=profile['local_size_jitter_max'],
            boundary_bias=profile['boundary_bias'],
            deformation_strength=profile['deformation_strength'],
            warp_affine_strength=profile['warp_affine_strength'],
            warp_radial_strength=profile['warp_radial_strength'],
            tet_quality_mode=tet_quality_mode,
            optimize_methods_csv=optimize_methods_csv,
            timeout_sec=timeout_sec,
        )
        node_counts.append(int(s['num_nodes']))
        tbad_counts.append(int(s['num_tets_bad']))
        netgen_flags.append(bool(s['netgen_used']))

    node_counts = np.asarray(node_counts, dtype=np.int32)
    tbad_counts = np.asarray(tbad_counts, dtype=np.int32)
    netgen_flags = np.asarray(netgen_flags, dtype=bool)

    return {
        'node_median': int(np.median(node_counts)),
        'tbad_median': int(np.median(tbad_counts)),
        'node_counts': node_counts,
        'tbad_counts': tbad_counts,
        'netgen_used_ratio': float(np.mean(netgen_flags)),
    }


def build_sphere_dataset_subprocess(
    *,
    out_path,
    n_samples,
    worker_path,
    radius,
    mesh_size_start,
    target_nodes,
    node_tolerance,
    target_tets,
    tet_tolerance_frac,
    max_attempts_per_sample,
    worker_retries,
    worker_timeout_sec,
    optimize_methods,
    seed,
    tet_quality_mode,
    diversity_profiles,
    auto_adjust_targets=True,
    calibration_samples=16,
    flush_every=10,
    gc_every=20,
):
    n_samples = int(n_samples)
    worker_path = Path(worker_path).resolve()
    if not worker_path.exists():
        raise FileNotFoundError(str(worker_path))

    optimize_methods_csv = ','.join([str(x) for x in optimize_methods])
    if 'Netgen' not in optimize_methods_csv and 'netgen' not in optimize_methods_csv.lower():
        raise ValueError('optimize_methods must include Netgen')

    # Calibrate realistic count targets from actual worker outputs.
    target_nodes_eff = int(target_nodes)
    target_tets_eff = int(target_tets)
    calib = None
    if auto_adjust_targets:
        calib = calibrate_count_targets(
            worker_path=worker_path,
            radius=radius,
            mesh_size_start=mesh_size_start,
            tet_quality_mode=tet_quality_mode,
            optimize_methods_csv=optimize_methods_csv,
            profile=diversity_profiles[0],
            timeout_sec=worker_timeout_sec,
            seed=seed,
            n_calib=calibration_samples,
        )

        # Target tets is for T_bad (padding-critical). If unset or unrealistic, adjust.
        if target_tets_eff <= 0:
            target_tets_eff = int(calib['tbad_median'])
        else:
            rel_err = abs(target_tets_eff - calib['tbad_median']) / max(1.0, float(calib['tbad_median']))
            if rel_err > max(0.05, float(tet_tolerance_frac)):
                target_tets_eff = int(calib['tbad_median'])

        if target_nodes_eff <= 0:
            target_nodes_eff = int(calib['node_median'])

    rng = np.random.default_rng(int(seed))
    f = init_mat73_stream_file(out_path, n_samples)

    node_counts = np.zeros((n_samples,), dtype=np.int32)
    tets_good_counts = np.zeros((n_samples,), dtype=np.int32)
    tets_bad_counts = np.zeros((n_samples,), dtype=np.int32)
    attempts_used = np.zeros((n_samples,), dtype=np.int32)
    retries_used = np.zeros((n_samples,), dtype=np.int32)
    profile_used = np.zeros((n_samples,), dtype=np.int32)
    matched_flags = np.zeros((n_samples,), dtype=bool)
    netgen_flags = np.zeros((n_samples,), dtype=bool)

    legacy_t_good_written = False
    fail_count = 0
    t0 = time.time()

    try:
        for i in tqdm(range(n_samples), desc='Generating samples (Netgen enforced)'):
            success = False
            last_err = None

            for r in range(int(worker_retries) + 1):
                for pidx, p in enumerate(diversity_profiles):
                    sample_seed = int(rng.integers(1, 2_000_000_000))
                    try:
                        sample = run_worker_sample(
                            worker_path,
                            seed=sample_seed,
                            radius=radius,
                            mesh_size_start=mesh_size_start,
                            target_nodes=target_nodes_eff,
                            node_tolerance=node_tolerance,
                            target_tets=target_tets_eff,
                            tet_tolerance_frac=tet_tolerance_frac,
                            max_attempts=max_attempts_per_sample,
                            n_embedded_points=p['n_embedded_points'],
                            embedded_radius_frac=p['embedded_radius_frac'],
                            local_size_jitter_min=p['local_size_jitter_min'],
                            local_size_jitter_max=p['local_size_jitter_max'],
                            boundary_bias=p['boundary_bias'],
                            deformation_strength=p['deformation_strength'],
                            warp_affine_strength=p['warp_affine_strength'],
                            warp_radial_strength=p['warp_radial_strength'],
                            tet_quality_mode=tet_quality_mode,
                            optimize_methods_csv=optimize_methods_csv,
                            timeout_sec=worker_timeout_sec,
                        )

                        if not sample['netgen_used']:
                            raise RuntimeError('netgen_not_used')
                        if not sample['matched']:
                            raise RuntimeError(
                                'sample_not_matched nodes={} t_bad={} t_good={} target_nodes={}±{} target_tets={}±{}%'.format(
                                    sample['num_nodes'],
                                    sample['num_tets_bad'],
                                    sample['num_tets_good'],
                                    target_nodes_eff,
                                    node_tolerance,
                                    target_tets_eff,
                                    int(round(100 * tet_tolerance_frac)),
                                )
                            )

                        success = True
                        retries_used[i] = r
                        profile_used[i] = pidx
                        break
                    except Exception as e:
                        last_err = 'retry={} profile={} err={}'.format(r, pidx, str(e))
                if success:
                    break

            if not success:
                fail_count += 1
                raise RuntimeError('sample {} failed after {} retries. last_error={}'.format(i, worker_retries, last_err))

            P = sample['P']
            T_bad_1 = sample['T_bad'] + 1
            T_good_1 = sample['T_good'] + 1

            write_one_sample(
                f,
                i,
                P,
                T_bad_1,
                T_good_1,
                sample['minQ_bad'],
                sample['minQ_good'],
            )

            if not legacy_t_good_written:
                f.create_dataset('T_good', data=np.asarray(T_good_1, dtype=np.int32).T)
                legacy_t_good_written = True

            node_counts[i] = int(sample['num_nodes'])
            tets_good_counts[i] = int(sample['num_tets_good'])
            tets_bad_counts[i] = int(sample['num_tets_bad'])
            attempts_used[i] = int(sample['attempts'])
            matched_flags[i] = bool(sample['matched'])
            netgen_flags[i] = bool(sample['netgen_used'])

            if (i + 1) % int(flush_every) == 0:
                f.flush()
            if (i + 1) % int(gc_every) == 0:
                gc.collect()

        elapsed = time.time() - t0
        params = {
            'dataset_type': 'sphere_gmsh_vs_delaunay_netgen_enforced',
            'n_samples': int(n_samples),
            'requested_target_nodes': int(target_nodes),
            'requested_target_tets': int(target_tets),
            'target_nodes': int(target_nodes_eff),
            'target_tets': int(target_tets_eff),
            'node_tolerance': int(node_tolerance),
            'tet_tolerance_frac': float(tet_tolerance_frac),
            'radius': float(radius),
            'mesh_size_start': float(mesh_size_start),
            'max_attempts_per_sample': int(max_attempts_per_sample),
            'worker_retries': int(worker_retries),
            'worker_timeout_sec': float(worker_timeout_sec),
            'optimize_methods': optimize_methods_csv,
            'seed': int(seed),
            'tet_quality_mode': str(tet_quality_mode),
            'auto_adjust_targets': int(1 if auto_adjust_targets else 0),
            'calibration_samples': int(calibration_samples),
            'calib_netgen_used_ratio': float(calib['netgen_used_ratio']) if calib is not None else np.nan,
            'calib_node_median': int(calib['node_median']) if calib is not None else -1,
            'calib_tbad_median': int(calib['tbad_median']) if calib is not None else -1,
            'matched_ratio': float(np.mean(matched_flags)),
            'netgen_used_ratio': float(np.mean(netgen_flags)),
            'node_count_min': int(np.min(node_counts)),
            'node_count_max': int(np.max(node_counts)),
            'node_count_mean': float(np.mean(node_counts)),
            'tet_good_min': int(np.min(tets_good_counts)),
            'tet_good_max': int(np.max(tets_good_counts)),
            'tet_good_mean': float(np.mean(tets_good_counts)),
            'tet_bad_min': int(np.min(tets_bad_counts)),
            'tet_bad_max': int(np.max(tets_bad_counts)),
            'tet_bad_mean': float(np.mean(tets_bad_counts)),
            'attempts_mean': float(np.mean(attempts_used)),
            'retries_mean': float(np.mean(retries_used)),
            'profile0_ratio': float(np.mean(profile_used == 0)),
            'profile1_ratio': float(np.mean(profile_used == 1)),
            'fail_count': int(fail_count),
            'elapsed_sec': float(elapsed),
        }

        write_params_group(f, params)
        f.flush()

        return {
            'out_path': out_path,
            'params': params,
            'node_counts': node_counts,
            'tets_good_counts': tets_good_counts,
            'tets_bad_counts': tets_bad_counts,
            'attempts_used': attempts_used,
            'retries_used': retries_used,
            'profile_used': profile_used,
            'matched_flags': matched_flags,
            'netgen_flags': netgen_flags,
            'minQ_bad': f['minQ_bad'][:, 0].copy(),
            'minQ_good': f['minQ_good'][:, 0].copy(),
        }
    finally:
        f.close()


In [51]:
# ---- Configuration ----
WORKER_PATH = 'sphere_sample_worker.py'
OUT_PATH = 'tet_dataset_sphere_targetTet734_targetNode164_N2000.mat'
N_SAMPLES = 2000

RADIUS = 1.0
MESH_SIZE_START = 0.28

TARGET_NODES = 164
NODE_TOLERANCE = 8
TARGET_TETS = 0  # auto-calibrate to realistic T_bad median
TET_TOLERANCE_FRAC = 0.05  # ±5%

MAX_ATTEMPTS_PER_SAMPLE = 10
WORKER_RETRIES = 10
WORKER_TIMEOUT_SEC = 150

SEED = 1
TET_QUALITY_MODE = 'mean_ratio'  # 'mean_ratio', 'simpqual1', 'simpqual2'

# Hard requirement: Netgen must be part of optimization methods.
OPTIMIZE_METHODS = ('Netgen',)

# Two high-diversity profiles. Both keep Netgen optimization; they only change
# embedded-point placement behavior to increase node-position diversity.
DIVERSITY_PROFILES = [
    # Netgen-stable profile: no embedded points; diversity comes from post-Netgen warps.
    dict(
        n_embedded_points=0,
        embedded_radius_frac=0.90,
        local_size_jitter_min=0.90,
        local_size_jitter_max=1.10,
        boundary_bias=0.30,
        deformation_strength=0.00,
        warp_affine_strength=0.14,
        warp_radial_strength=0.24,
    ),
    dict(
        n_embedded_points=0,
        embedded_radius_frac=0.90,
        local_size_jitter_min=0.90,
        local_size_jitter_max=1.10,
        boundary_bias=0.30,
        deformation_strength=0.00,
        warp_affine_strength=0.22,
        warp_radial_strength=0.34,
    ),
]


In [52]:
dataset_info = build_sphere_dataset_subprocess(
    out_path=OUT_PATH,
    n_samples=N_SAMPLES,
    worker_path=WORKER_PATH,
    radius=RADIUS,
    mesh_size_start=MESH_SIZE_START,
    target_nodes=TARGET_NODES,
    node_tolerance=NODE_TOLERANCE,
    target_tets=TARGET_TETS,
    tet_tolerance_frac=TET_TOLERANCE_FRAC,
    max_attempts_per_sample=MAX_ATTEMPTS_PER_SAMPLE,
    worker_retries=WORKER_RETRIES,
    worker_timeout_sec=WORKER_TIMEOUT_SEC,
    optimize_methods=OPTIMIZE_METHODS,
    seed=SEED,
    tet_quality_mode=TET_QUALITY_MODE,
    diversity_profiles=DIVERSITY_PROFILES,
    auto_adjust_targets=True,
    calibration_samples=16,
    flush_every=10,
    gc_every=20,
)

print('Saved:', dataset_info['out_path'])
print('matched_ratio:', dataset_info['params']['matched_ratio'])
print('netgen_used_ratio:', dataset_info['params']['netgen_used_ratio'])
print('node_count min/mean/max:', dataset_info['params']['node_count_min'], dataset_info['params']['node_count_mean'], dataset_info['params']['node_count_max'])
print('tet_good min/mean/max:', dataset_info['params']['tet_good_min'], dataset_info['params']['tet_good_mean'], dataset_info['params']['tet_good_max'])
print('tet_bad  min/mean/max:', dataset_info['params']['tet_bad_min'], dataset_info['params']['tet_bad_mean'], dataset_info['params']['tet_bad_max'])
print('attempts_mean:', dataset_info['params']['attempts_mean'], 'retries_mean:', dataset_info['params']['retries_mean'])
print('profile0_ratio:', dataset_info['params']['profile0_ratio'], 'profile1_ratio:', dataset_info['params']['profile1_ratio'])
print('minQ_bad mean:', float(np.nanmean(dataset_info['minQ_bad'])))
print('minQ_good mean:', float(np.nanmean(dataset_info['minQ_good'])))
print('elapsed_sec:', dataset_info['params']['elapsed_sec'])


Generating samples (Netgen enforced):   0%|          | 0/2000 [00:00<?, ?it/s]

RuntimeError: sample 0 failed after 10 retries. last_error=retry=10 profile=1 err=sample_not_matched nodes=213 t_bad=865 t_good=694 target_nodes=164±8 target_tets=899±5%

In [ ]:
# Loader sanity check for compatibility with current training pipeline.
from tet_mat73_loader import TetMat73Dataset

ds = TetMat73Dataset(OUT_PATH, load_all=False)
print('num_samples:', ds.num_samples)
print('legacy T_good shape:', ds.T_good.shape)
print('minQ_bad shape:', ds.minQ_bad.shape)
print('minQ_good shape:', ds.minQ_good.shape)
P0, Tbad0 = ds.get(0)
print('sample[0] P shape:', P0.shape, 'T_bad shape:', Tbad0.shape)
ds.close()


## 1) Tet Quality Distribution Comparison (Bad vs Good)

"
"Compares all tet-quality values across the dataset in one overlaid plot.


In [ ]:
import h5py
from tet_quality_metrics import compute_tet_quality


def _cell_ref(cell_ds, i):
    if cell_ds.shape[0] == 1:
        return cell_ds[0, i]
    if cell_ds.shape[1] == 1:
        return cell_ds[i, 0]
    r, c = cell_ds.shape
    return cell_ds[i % r, i // r]


def _read_str_param(f, key, default='mean_ratio'):
    if 'params' not in f or key not in f['params']:
        return default
    raw = f['params'][key][()]
    if isinstance(raw, bytes):
        return raw.decode('utf-8')
    arr = np.asarray(raw).reshape(-1)
    if arr.size == 0:
        return default
    v = arr[0]
    if isinstance(v, bytes):
        return v.decode('utf-8')
    return str(v)


with h5py.File(OUT_PATH, 'r') as f:
    n_samples = int(max(f['P'].shape))
    mode = _read_str_param(f, 'tet_quality_mode', TET_QUALITY_MODE)
    if mode not in ('mean_ratio', 'simpqual1', 'simpqual2'):
        mode = 'mean_ratio'

    has_good_cell = 'T_good_cell' in f
    T_good_global = (np.array(f['T_good']).T - 1).astype(np.int32) if not has_good_cell else None

    q_bad_all = []
    q_good_all = []
    P_list = []
    node_counts = []
    good_tet_counts = []

    for i in tqdm(range(n_samples), desc='Load + quality'):
        p_ref = _cell_ref(f['P'], i)
        tb_ref = _cell_ref(f['T_bad'], i)
        P = np.array(f[p_ref]).T.astype(np.float64)
        T_bad = (np.array(f[tb_ref]).T - 1).astype(np.int32)

        if has_good_cell:
            tg_ref = _cell_ref(f['T_good_cell'], i)
            T_good = (np.array(f[tg_ref]).T - 1).astype(np.int32)
        else:
            T_good = T_good_global

        q_bad = compute_tet_quality(P, T_bad, mode=mode)
        q_good = compute_tet_quality(P, T_good, mode=mode)

        q_bad_all.append(q_bad.astype(np.float32, copy=False))
        q_good_all.append(q_good.astype(np.float32, copy=False))
        P_list.append(P)
        node_counts.append(P.shape[0])
        good_tet_counts.append(T_good.shape[0])

q_bad_all = np.concatenate(q_bad_all, axis=0)
q_good_all = np.concatenate(q_good_all, axis=0)
node_counts = np.asarray(node_counts, dtype=np.int32)
good_tet_counts = np.asarray(good_tet_counts, dtype=np.int32)

q_min = float(min(np.min(q_bad_all), np.min(q_good_all)))
q_max = float(max(np.max(q_bad_all), np.max(q_good_all)))
bins = np.linspace(q_min, q_max, 140)

plt.figure(figsize=(10, 5.5))
plt.hist(q_bad_all, bins=bins, density=True, alpha=0.45, color='#d95f02', label='Bad mesh (Delaunay)')
plt.hist(q_good_all, bins=bins, density=True, alpha=0.45, color='#1b9e77', label='Good mesh (Netgen optimized)')
plt.title('All-Tet Quality Distribution: Bad vs Good')
plt.xlabel('Tet quality')
plt.ylabel('Density')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print('quality_mode:', mode)
print('num_samples:', len(P_list))
print('total_tets_bad:', int(q_bad_all.size), 'total_tets_good:', int(q_good_all.size))
print('bad mean/min:', float(np.mean(q_bad_all)), float(np.min(q_bad_all)))
print('good mean/min:', float(np.mean(q_good_all)), float(np.min(q_good_all)))
print('good tet count min/mean/max:', int(np.min(good_tet_counts)), float(np.mean(good_tet_counts)), int(np.max(good_tet_counts)))


## 2) Node Position Diversity Analysis

"
"Measures node-position diversity while checking element-count compactness.


In [ ]:
def symmetric_chamfer_l2(a, b):
    d2 = np.sum((a[:, None, :] - b[None, :, :]) ** 2, axis=2)
    return float(np.sqrt(np.mean(np.min(d2, axis=1)) + np.mean(np.min(d2, axis=0))))


all_r_max = float(max(np.max(np.linalg.norm(P - P.mean(axis=0, keepdims=True), axis=1)) for P in P_list))
rad_bins = np.linspace(0.0, all_r_max, 26)
rad_centers = 0.5 * (rad_bins[:-1] + rad_bins[1:])

rad_hists = []
for P in P_list:
    c = P.mean(axis=0, keepdims=True)
    r = np.linalg.norm(P - c, axis=1)
    h, _ = np.histogram(r, bins=rad_bins, density=True)
    rad_hists.append(h)
rad_hists = np.asarray(rad_hists, dtype=np.float64)
rad_mean = np.mean(rad_hists, axis=0)
rad_std = np.std(rad_hists, axis=0)
rad_div_score = float(np.mean(rad_std))

rng = np.random.default_rng(0)
n = len(P_list)
num_pairs = min(350, n * (n - 1) // 2)
seen = set()
pairs = []
while len(pairs) < num_pairs:
    i = int(rng.integers(0, n))
    j = int(rng.integers(0, n))
    if i == j:
        continue
    a, b = (i, j) if i < j else (j, i)
    if (a, b) in seen:
        continue
    seen.add((a, b))
    pairs.append((a, b))

chamfers = np.zeros((len(pairs),), dtype=np.float64)
for k, (i, j) in enumerate(tqdm(pairs, desc='Pairwise Chamfer')):
    chamfers[k] = symmetric_chamfer_l2(P_list[i], P_list[j])

fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))

axes[0].hist(node_counts, bins=min(30, max(8, len(np.unique(node_counts)))), color='#7570b3', alpha=0.9)
axes[0].set_title('Node Count Distribution')
axes[0].set_xlabel('#nodes')
axes[0].set_ylabel('Count')
axes[0].grid(alpha=0.25)

axes[1].hist(good_tet_counts, bins=min(30, max(8, len(np.unique(good_tet_counts)))), color='#66a61e', alpha=0.9)
axes[1].set_title('Good Tet Count Distribution')
axes[1].set_xlabel('#good tets')
axes[1].set_ylabel('Count')
axes[1].grid(alpha=0.25)

axes[2].plot(rad_centers, rad_mean, color='#1b9e77', lw=2, label='mean')
axes[2].fill_between(rad_centers, rad_mean - rad_std, rad_mean + rad_std, color='#1b9e77', alpha=0.25, label='±1 std')
axes[2].set_title('Radial Signature Variability')
axes[2].set_xlabel('Radius from centroid')
axes[2].set_ylabel('Density')
axes[2].legend()
axes[2].grid(alpha=0.25)

axes[3].hist(chamfers, bins=30, color='#e7298a', alpha=0.9)
axes[3].set_title('Pairwise Chamfer Distance')
axes[3].set_xlabel('Chamfer L2')
axes[3].set_ylabel('Count')
axes[3].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print('node_count mean/std/min/max:', float(np.mean(node_counts)), float(np.std(node_counts)), int(np.min(node_counts)), int(np.max(node_counts)))
print('good_tet_count mean/std/min/max:', float(np.mean(good_tet_counts)), float(np.std(good_tet_counts)), int(np.min(good_tet_counts)), int(np.max(good_tet_counts)))
print('good_tet_relative_span:', float((np.max(good_tet_counts) - np.min(good_tet_counts)) / max(1.0, np.mean(good_tet_counts))))
print('radial_diversity_score (mean std over bins):', rad_div_score)
print('chamfer mean/std/p10/p50/p90:',
      float(np.mean(chamfers)),
      float(np.std(chamfers)),
      float(np.percentile(chamfers, 10)),
      float(np.percentile(chamfers, 50)),
      float(np.percentile(chamfers, 90)))
